In [55]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import matplotlib
from matplotlib import colors
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import FormatStrFormatter
from matplotlib import cm

import h5py
import numpy as np

%matplotlib widget

In [56]:
def Shen_fit_uncer(z, lums): ###best fit data from Shen+2020

    def get_params():
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params

    def shen_func(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)
        return np.log10(Phibol)

    params = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    param_list = np.array([params[i] for i in params])

    NUM = int(1e4)

    rand_params = get_params()
    ys = np.apply_along_axis(shen_func, 1, rand_params).T
    ya = shen_func(param_list[:,0])

    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya
    
    zr = 2.0
    zfrac = (1 + z)/(1 + zr)
    logLs = 2*params['c0'][0]/(zfrac**params['c1'][0] + zfrac**params['c2'][0])

    return ya, std_ave, std_abv, std_blw, logLs


In [57]:
from numpy.polynomial import chebyshev as C
import numpy as np
import scipy as sp
import scipy.stats
import glob


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
ssfr_a_list = np.array([float(a) for a in files])
ssfr_mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]

files = [f.split('a')[1].split('.d')[0] for f in glob.glob('smfs/smf_a*.dat')]
smf_a_list = np.array([float(a) for a in files])
smf_mass_list = np.array([np.loadtxt("smfs/smf_a"+f+".dat")[:,0] for f in files])
smf_list = [np.loadtxt("smfs/smf_a"+f+".dat")[:,1] for f in files]

param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

In [58]:
class QLF():
    def __init__(self, z, bins):


        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bins = bins

        self.StellBins = np.linspace(3.0, 12.5, int((12.5 - 3.0) / self.bins))
        
        #### extract ssfr
        closest_a = np.argmin(np.abs(ssfr_a_list - self.a))
        ssfrs = np.array(ssfr_list[closest_a])
        nonzero = (ssfrs != 0)
        masses = np.array(ssfr_mass_list[closest_a])
        self.SSFRs = 10**np.interp(self.StellBins, masses[nonzero], np.log10(ssfrs[nonzero]))
        
        #### extrapolate out to high M*
        slope = (np.log10(ssfrs[nonzero][-1]) - np.log10(ssfrs[nonzero][-2])) / (masses[nonzero][-1] - masses[nonzero][-2])
        inter = np.log10(ssfrs[nonzero][-1]) - slope * masses[nonzero][-1]
        gtzero = (self.StellBins >= masses[nonzero][-1])
        self.SSFRs[gtzero] = 10**(self.StellBins[gtzero]*slope + inter)
        
        #### extrapolate out to low M*
        slope = (np.log10(ssfrs[1]) - np.log10(ssfrs[0])) / (masses[1] - masses[0])
        inter = np.log10(ssfrs[0]) - slope * masses[0]
        ltavail = (self.StellBins < masses[0])
        self.SSFRs[ltavail] = 10**(self.StellBins[ltavail]*slope + inter)
        
        
        ### extract smf
        closest_a = np.argmin(np.abs(smf_a_list - self.a))
        smf = np.array(smf_list[closest_a])
        nonzero = (smf != 0)
        masses = np.array(smf_mass_list[closest_a])
        self.dNdlogMstar = 10**np.interp(self.StellBins, masses[nonzero], np.log10(smf[nonzero]))
        
        #### extrapolate out to high M*
        slope = (np.log10(smf[nonzero][-1]) - np.log10(smf[nonzero][-2])) / (masses[nonzero][-1] - masses[nonzero][-2])
        inter = np.log10(smf[nonzero][-1]) - slope * masses[nonzero][-1]
        self.dNdlogMstar[gtzero] = 10**(self.StellBins[gtzero]*slope + inter)
        
        #### extrapolate out to low M*
        slope = (np.log10(smf[1]) - np.log10(smf[0])) / (masses[1] - masses[0])
        inter = np.log10(smf[0]) - slope * masses[0]
        self.dNdlogMstar[ltavail] = 10**(self.StellBins[ltavail]*slope + inter)
        
        self.dNdlnMstar = self.dNdlogMstar/np.log(10)
 ############################       
        ##### bulge calculations
        def BT_sigmoid(logMstar): #input log10 of Mstar in units of solar mass 
            return 1.0 / (1.0 + np.exp(-1.6*(logMstar-10.02))) #this is B/T

        BT = BT_sigmoid(self.StellBins)
        self.BulgeBins = np.log10(BT * 10**self.StellBins) #this is bulge mass in units of solar mass
        dBTdMstar = np.gradient(BT, 10**self.StellBins)
        dMbdMstar = dBTdMstar*10**self.StellBins+BT
        self.SMBARs = (self.SSFRs*dMbdMstar*10**self.StellBins) / 10**self.BulgeBins #this is units of per year
        
        dlogMstardlogMb = np.gradient(self.StellBins, self.BulgeBins)
        self.dNdlogMbulge = self.dNdlogMstar*dlogMstardlogMb
        self.dNdlnMbulge = self.dNdlogMbulge/np.log(10)


    def get_Mbh(self, logMb0, slope_low = 0.2, norm_from_local =4.0, norm_local = 8.2):
        norm = [11, norm_local]
        
        self.logMb0 = logMb0
        logMb = self.BulgeBins
        logMbh = logMb * 0
        slopes = logMb * 0

        logMbh0 = logMb0 + norm[1] - norm[0] - norm_from_local
        my_norm = [logMb0, logMbh0]
        post_params = [1, logMbh0 - logMb0]
        pre_params = [slope_low, my_norm[1] - my_norm[0] * slope_low]

        post = (logMb > logMb0)
        pre = (logMb <= logMb0)
        Mb = 10**logMb
        Mb0 = 10**logMb0
        Mbh0 = 10**logMbh0
        beta = post_params[0]
        alpha = 10**norm[1] / 10**(norm[0]*beta)

        Mbh = Mbh0 + alpha * Mb[post]**(beta-1) * (Mb[post] - Mb0)
        logMbh[post] = np.log10(Mbh)
        slopes[post] = alpha * beta * Mb[post]**beta / Mbh

        logMbh[pre] = logMb[pre] * pre_params[0] + pre_params[1]
        slopes[pre] = pre_params[0]

        
        self.slopes = slopes
        self.BHBins = logMbh
        self.pre = pre
        self.post = post
        ## SSFR are in yr^-1
        self.SBHARs = self.slopes * (self.SMBARs / 3.154e7) ## s^-1
        self.MdotBH = self.SBHARs * (10**self.BHBins * 2e33) ## g/s

#############################

    def get_Mdotbh(self, vals, files = files):

        lnxsig = vals[0]
        Mdotbh = vals[1]
        
        mu_lnX = -0.5 * lnxsig**2
        mu_lnMdotbh = mu_lnX + np.log(Mdotbh) #g/s
        
        return mu_lnMdotbh, lnxsig


    def gauss_Mdot(self, lnMdotbh):

        x = lnMdotbh
        mu = self.Mdot_mu_sig[:,0]
        sig = self.Mdot_mu_sig[:,1]
        y = ( 1/np.sqrt(2.0 * np.pi * sig*sig) ) * np.exp( -(x - mu)**2.0 / (2.0 * sig*sig) )

        return y


    def get_dNdlnL(self, L, lnxsigs): #input luminosity in log10 space in units of solar mass
        
        
        
        ###############################
        lnxsig_list = self.StellBins * 0
        lnxsig_list[self.pre] = lnxsigs[0]
        lnxsig_list[self.post] = lnxsigs[1]
        ### start transition at the M*crit value
        critpoint = np.argmin(np.abs(self.BulgeBins - self.logMb0))
        ### end transistion 0.5 dex after that value
        endtran = np.argmin(np.abs(self.BulgeBins - (self.logMb0 + 0.5)))
        lintrans = np.linspace(lnxsigs[0], lnxsigs[1], len(lnxsig_list[critpoint-1:endtran]))
        lnxsig_list[critpoint-1:endtran] = lintrans
        ################################
        


        vals = np.zeros((len(self.StellBins), 2))
        vals[:,0] = lnxsig_list
        vals[:,1] = self.MdotBH
        self.Mdot_mu_sig = np.apply_along_axis(self.get_Mdotbh, 1, vals)

        self.lnMdotbh_list = (np.asarray(L) + np.log10(3.9e33)) * np.log(10) - np.log(0.1*2.99e10**2)

        ### I convert the stellar bins size to log base e for the calculation with the log base e stellar mass function and log base e sigmas
        self.intvals = np.apply_along_axis(self.gauss_Mdot, 1, self.lnMdotbh_list.reshape(len(self.lnMdotbh_list),1)) * self.dNdlnMstar * (self.StellBins[1] - self.StellBins[0]) * np.log(10)
     
        #### I produce two QLFs one in log base e and the other in log base 10
        self.dNdlnL = np.sum(self.intvals, axis = 1)
        self.dNdlogL = self.dNdlnL * np.log(10)



In [59]:
class QLF_orig():
    def __init__(self, z, bins):


        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bins = bins

        self.StellBins = np.linspace(3.0, 12.5, int((12.5 - 3.0) / self.bins))
        
        #### extract ssfr
        closest_a = np.argmin(np.abs(ssfr_a_list - self.a))
        ssfrs = np.array(ssfr_list[closest_a])
        nonzero = (ssfrs != 0)
        masses = np.array(ssfr_mass_list[closest_a])
        self.SSFRs = 10**np.interp(self.StellBins, masses[nonzero], np.log10(ssfrs[nonzero]))
        
        #### extrapolate out to high M*
        slope = (np.log10(ssfrs[nonzero][-1]) - np.log10(ssfrs[nonzero][-2])) / (masses[nonzero][-1] - masses[nonzero][-2])
        inter = np.log10(ssfrs[nonzero][-1]) - slope * masses[nonzero][-1]
        gtzero = (self.StellBins >= masses[nonzero][-1])
        self.SSFRs[gtzero] = 10**(self.StellBins[gtzero]*slope + inter)
        
        #### extrapolate out to low M*
        slope = (np.log10(ssfrs[1]) - np.log10(ssfrs[0])) / (masses[1] - masses[0])
        inter = np.log10(ssfrs[0]) - slope * masses[0]
        ltavail = (self.StellBins < masses[0])
        self.SSFRs[ltavail] = 10**(self.StellBins[ltavail]*slope + inter)
        
        
        ### extract smf
        closest_a = np.argmin(np.abs(smf_a_list - self.a))
        smf = np.array(smf_list[closest_a])
        nonzero = (smf != 0)
        masses = np.array(smf_mass_list[closest_a])
        self.dNdlogMstar = 10**np.interp(self.StellBins, masses[nonzero], np.log10(smf[nonzero]))
        
        #### extrapolate out to high M*
        slope = (np.log10(smf[nonzero][-1]) - np.log10(smf[nonzero][-2])) / (masses[nonzero][-1] - masses[nonzero][-2])
        inter = np.log10(smf[nonzero][-1]) - slope * masses[nonzero][-1]
        self.dNdlogMstar[gtzero] = 10**(self.StellBins[gtzero]*slope + inter)
        
        #### extrapolate out to low M*
        slope = (np.log10(smf[1]) - np.log10(smf[0])) / (masses[1] - masses[0])
        inter = np.log10(smf[0]) - slope * masses[0]
        self.dNdlogMstar[ltavail] = 10**(self.StellBins[ltavail]*slope + inter)
        
        self.dNdlnMstar = self.dNdlogMstar/np.log(10)
        


    def get_Mbh(self, logMstar0, slope_low = 0.2, norm_from_local =4.0, approx_local = False, norm_local = 8.2):
        norm = [11, norm_local]
        
        self.logMstar0 = logMstar0
        logMstar = self.StellBins
        logMbh = logMstar * 0
        slopes = logMstar * 0

        if approx_local == False:
            logMbh0 = logMstar0*1.12 - norm[0]*1.12 + norm[1] - norm_from_local
            my_norm = [logMstar0, logMbh0]
            post_params = [1.12, logMbh0 - 1.12*logMstar0]
            pre_params = [slope_low, my_norm[1] - my_norm[0] * slope_low]

        else:
            logMbh0 = logMstar0 + norm[1] - norm[0] - norm_from_local
            my_norm = [logMstar0, logMbh0]
            post_params = [1, logMbh0 - logMstar0]
            pre_params = [slope_low, my_norm[1] - my_norm[0] * slope_low]

        post = (logMstar > logMstar0)
        pre = (logMstar <= logMstar0)
        Ms = 10**logMstar
        Ms0 = 10**logMstar0
        Mbh0 = 10**logMbh0
        beta = post_params[0]
        alpha = 10**norm[1] / 10**(norm[0]*beta)

        Mbh = Mbh0 + alpha * Ms[post]**(beta-1) * (Ms[post] - Ms0)
        logMbh[post] = np.log10(Mbh)
        slopes[post] = alpha * beta * Ms[post]**beta / Mbh

        logMbh[pre] = logMstar[pre] * pre_params[0] + pre_params[1]
        slopes[pre] = pre_params[0]

        
        self.slopes = slopes
        self.BHBins = logMbh
        self.pre = pre
        self.post = post
        self.mmax = post_params[0]
        ## SSFR are in yr^-1
        self.SBHARs = self.slopes * (self.SSFRs / 3.154e7) ## s^-1
        self.MdotBH = self.SBHARs * (10**self.BHBins * 2e33) ## g/s
        self.Ledd = 1.3e38 * 10**self.BHBins ## erg/s
        self.Mdotedd = self.Ledd / (.1 * (2.99e10)**2) ## g/s


    def get_Mdotbh(self, vals, files = files):

        lnxsig = vals[0]
        Mdotbh = vals[1]
        
        mu_lnX = -0.5 * lnxsig**2
        mu_lnMdotbh = mu_lnX + np.log(Mdotbh) #g/s
        
        return mu_lnMdotbh, lnxsig


    def gauss_Mdot(self, lnMdotbh):

        x = lnMdotbh
        mu = self.Mdot_mu_sig[:,0]
        sig = self.Mdot_mu_sig[:,1]
        y = ( 1/np.sqrt(2.0 * np.pi * sig*sig) ) * np.exp( -(x - mu)**2.0 / (2.0 * sig*sig) )

        return y


    def get_dNdlnL(self, L, lnxsigs): #input luminosity in log10 space in units of solar mass
        
        lnxsig_list = self.StellBins * 0
        lnxsig_list[self.pre] = lnxsigs[0]
        lnxsig_list[self.post] = lnxsigs[1]
        ### start transition at the M*crit value
        critpoint = np.argmin(np.abs(self.StellBins - self.logMstar0))
        ### end transistion 0.5 dex after that value
        endtran = np.argmin(np.abs(self.StellBins - (self.logMstar0 + 0.5)))
        lintrans = np.linspace(lnxsigs[0], lnxsigs[1], len(lnxsig_list[critpoint-1:endtran]))
        lnxsig_list[critpoint-1:endtran] = lintrans
        


        vals = np.zeros((len(self.StellBins), 2))
        vals[:,0] = lnxsig_list
        vals[:,1] = self.MdotBH
        self.Mdot_mu_sig = np.apply_along_axis(self.get_Mdotbh, 1, vals)

        self.lnMdotbh_list = (np.asarray(L) + np.log10(3.9e33)) * np.log(10) - np.log(0.1*2.99e10**2)

        ### I convert the stellar bins size to log base e for the calculation with the log base e stellar mass function and log base e sigmas
        self.intvals = np.apply_along_axis(self.gauss_Mdot, 1, self.lnMdotbh_list.reshape(len(self.lnMdotbh_list),1)) * self.dNdlnMstar * (self.StellBins[1] - self.StellBins[0]) * np.log(10)
     
        #### I produce two QLFs one in log base e and the other in log base 10
        self.dNdlnL = np.sum(self.intvals, axis = 1)
        self.dNdlogL = self.dNdlnL * np.log(10)



## Alex R's fit vs the function I fit.

In [6]:
def BT_alex_fit(Mstar): #Mstar must be in units of solar mass
    BT = 0.4240*(Mstar/1e10)**0.3887 #this is the ratio of bulge stellar mass to total stellar mass
    BT[Mstar < 7e7] = 0
    BT[Mstar > 7e10] = 1
    return BT

def fsigmoid(x, a, b):
    return 1.0 / (1.0 + np.exp(-a*(x-b)))

### Symmetric sigmoid function fit.

In [86]:
%matplotlib widget

both = {'x':[], 'y':[], 'yerrup':[], 'yerrdown':[]}

benson = {'x':[], 'y':[], 'yerrup':[], 'yerrdown':[]}
f = open('plot_data/Benson+2017.txt', 'r')
for line in f:
    if line[0] != '#':
        dat = line.split(', ')
        dat[-1] = dat[-1].split('\n')[0]
        dat = [float(d) for d in dat]
        benson['x'].append(float(dat[0]))
        benson['y'].append(float(dat[1]))
        both['x'].append(float(dat[0]))
        both['y'].append(float(dat[1]))
        benson['yerrup'].append(float(dat[2])-float(dat[1]))
        benson['yerrdown'].append(float(dat[1])-float(dat[3]))
        both['yerrup'].append(float(dat[2])-float(dat[1]))
        both['yerrdown'].append(float(dat[1])-float(dat[3]))
        
salo = {'x':[], 'y':[]}
f = open('plot_data/Salo+2015.txt', 'r')
for line in f:
    if line[0] != '#':
        dat = line.split(', ')
        dat[-1] = dat[-1].split('\n')[0]
        dat = [float(d) for d in dat]
        salo['x'].append(float(dat[0]))
        salo['y'].append(float(dat[1]))
        both['x'].append(float(dat[0]))
        both['y'].append(float(dat[1]))
        both['yerrup'].append(0)
        both['yerrdown'].append(0)
        
        
qlf = QLF(1.0, 0.005)
xdata_model = qlf.StellBins
ydata_model = BT_alex_fit(10**xdata_model)

# params, pcov = sp.optimize.curve_fit(fsigmoid, xdata_model, ydata_model, method='dogbox', p0 = [1,9], bounds = ([1,5],[3,10.5]))

xdata_obs = both['x']
ydata_obs = both['y']
params, pcov = sp.optimize.curve_fit(fsigmoid, xdata_obs, ydata_obs, method='dogbox', p0 = [1,9], bounds = ([1,5],[3,10.5]))
plt.plot(xdata_model, fsigmoid(xdata_model, params[0], params[1]), label='B1', c='k')
print('Obs data params:',params)

# xdata_obs = both['x']
# ydata_obs = np.zeros(len(both['y'])) + np.asarray(both['y']) + np.asarray(both['yerrup'])
# params, pcov = sp.optimize.curve_fit(fsigmoid, xdata_obs, ydata_obs, method='dogbox', p0 = [1,9], bounds = ([1,5],[15,10.5]))
# plt.plot(xdata_model, fsigmoid(xdata_model, params[0], params[1]), label='Sigmoid fit (upper bound).', c='k', ls = 'dashed')
# print('Upper bound params:',params)

xdata_obs = both['x']
ydata_obs = np.zeros(len(both['y'])) + np.asarray(both['y']) + np.asarray(both['yerrup'])
params, pcov = sp.optimize.curve_fit(fsigmoid, xdata_obs, ydata_obs, method='dogbox', p0 = [1,9], bounds = ([1,5],[2.0,10.5]))
plt.plot(xdata_model, fsigmoid(xdata_model, params[0], params[1]), label='B2', c='blue', ls = 'dashed')
print('Upper bound params:',params)

xdata_obs = both['x']
ydata_obs = np.zeros(len(both['y'])) + np.asarray(both['y']) - np.asarray(both['yerrdown'])
params, pcov = sp.optimize.curve_fit(fsigmoid, xdata_obs, ydata_obs, method='dogbox', p0 = [1,9], bounds = ([0,5],[3,15]))
plt.plot(xdata_model, fsigmoid(xdata_model, params[0], params[1]), label='B3', c='k', ls = 'dotted')
print('Lower bound params:',params)

# plt.plot(xdata_model, ydata_model, label="Alex's fit.", c='b')
# plt.plot(xdata_model, fsigmoid(xdata_model, 1.6, 10.02), label='Preliminary test.', c='gray')
# plt.plot(xdata_model, fsigmoid(xdata_model, 9, 9), label='test', c='gray')


plt.errorbar(benson['x'], benson['y'], yerr = [benson['yerrdown'], benson['yerrup']], c='r', fmt='.', label='Benson+2017', capsize=3)
plt.scatter(salo['x'], salo['y'], c='k', marker='x', label='Salo+2015 (median)')

plt.legend()

plt.axvline(10, c='k', linestyle='dotted')

plt.xlabel(r'$\log_{10}[\rmM_*/M_\odot]$')
plt.ylabel('B/T')
plt.xlim([5,12.5])

FigureCanvasNbAgg()

Obs data params: [ 1.63126656 10.08708065]
Upper bound params: [2.         8.98699947]
Lower bound params: [ 1.38000186 11.90976007]


(5, 12.5)

### Exploring also an asymmetric sigmoid function fit.

In [70]:
%matplotlib widget

qlf = QLF(1.0, 0.005)
xdata = qlf.StellBins
ydata = BT_alex_fit(10**xdata)

plt.plot(xdata, ydata, label="Alex's fit.")

def fsigmoid(x, a, b):
    return 1.0 / (1.0 + np.exp(a*(x+b)))

params, pcov = sp.optimize.curve_fit(fsigmoid, xdata, ydata, method='lm', p0 = [-1, -15])
print(params)

plt.plot(xdata, fsigmoid(xdata, params[0], params[1]), label='Sigmoid function fit 1.')


def fsigmoid2(x, a, b, c, d):
    return c + (d - c) / (1.0 + np.exp(a*(x+b)))

params2, pcov2 = sp.optimize.curve_fit(fsigmoid2, xdata, ydata, method='lm', p0 = [-1,-15, -5, -5])
print(params2)

plt.plot(xdata, fsigmoid2(xdata, params2[0], params2[1], params2[2], params2[3]), label='Sigmoid function fit 2.')
plt.plot(xdata, fsigmoid2(xdata, params2[0], params2[1], params2[2], params2[3]), label='Sigmoid function fit 2.')


plt.legend()

plt.xlabel(r'$\log_{10}[\rmM_*/M_\odot]$')
plt.ylabel('B/T')
plt.xlim([5,12])

plt.text(6,0.4, r'$f_{s2}(x) = c + \frac{d-c}{1+e^{a(x+b)}}$')
plt.text(6,0.6, r'$f_{s1}(x) = \frac{1}{1+e^{a(x+b)}}$')
plt.text(6,0.5, 'vs')

FigureCanvasNbAgg()

[ -1.96738768 -10.0231044 ]
[-1.78341243e+00 -1.01296648e+01  7.33573085e-03  1.06910530e+00]


Text(6,0.5,'vs')

## Implementing the bulge relation calculations.

### Figuring out the bulge Mdot values and the bulge mass function.

In [2]:
%matplotlib widget

fig = plt.figure(figsize=(10,4))
gs = gridspec.GridSpec(1, 2)

def BT_sigmoid(logMstar): #input log10 of Mstar in units of solar mass 
    return 1.0 / (1.0 + np.exp(-1.6*(logMstar-10.02))) #this is B/T

qlf = QLF(1.0, 0.005)
logMstar = qlf.StellBins
sSFR = qlf.SSFRs

BT = BT_sigmoid(logMstar)
Mbulge = BT * 10**logMstar #this is bulge mass in units of solar mass
dBTdMstar = np.gradient(BT, 10**logMstar)
dMbdMstar = dBTdMstar*10**logMstar+BT
dMbdt = sSFR*dMbdMstar*10**logMstar #this is units of solar mass per year


ax1 = fig.add_subplot(gs[0, 0])

ax1.plot(10**logMstar, sSFR, c='r', label=r'$\rmM=M_*$')
ax1.plot(Mbulge, dMbdt/Mbulge, c='b', label=r'$\rmM=M_{bulge}$')

ax1.set_xlabel(r'M ($\rmM_\odot$)')
ax1.set_ylabel(r'dM/dt ($\rmM_\odot yr^{-1}$)')
ax1.set_xscale('log')
ax1.set_xlim([1e7, 1e12])
ax1.legend()






dNdlogMstar = qlf.dNdlogMstar
dlogMstardlogMb = np.gradient(logMstar, np.log10(Mbulge))
dNdlogMbulge = dNdlogMstar*dlogMstardlogMb

ax2 = fig.add_subplot(gs[0, 1])

ax2.plot(10**logMstar, np.log10(dNdlogMstar), c='r', label=r'$\rmM=M_*$')
ax2.plot(Mbulge, np.log10(dNdlogMbulge), c='b', label=r'$\rmM=M_{bulge}$')

ax2.set_xlabel(r'M ($\rmM_\odot$)')
ax2.set_ylabel(r'$\log_{10} \Phi (\rmMpc^{-3} \log_{10} [M]^{-1})$',fontsize=12)
ax2.set_xscale('log')
ax2.set_xlim([1e7, 1e12])
ax2.set_ylim([-7,0])
ax2.legend()

FigureCanvasNbAgg()

NameError: name 'QLF' is not defined

### Testing the implimentation to the QLF.

In [59]:
plt.close('all')
fig = plt.figure(figsize=(14,8))
gs = gridspec.GridSpec(2, 3)

pre, post, Mb_crit, b = 1.86, 1.86, 10.7, 0.005
slope_low, norm_from_local, norm_of_local = .6, 2.0, -2.8
L = np.linspace(5,18,200)
z, b = 1.0, 0.005

qlf = QLF(z, b)
qlf.get_Mbh(logMb0 = Mb_crit, slope_low = slope_low, norm_from_local = norm_from_local, norm_local = 11+norm_of_local)
qlf.get_dNdlnL(L, [pre, post])

logMstar = qlf.StellBins
logMbulge = qlf.BulgeBins
logMbh = qlf.BHBins
SSFRs = qlf.SSFRs
SMBARs = qlf.SMBARs

dNdlogMstar = qlf.dNdlogMstar
dNdlogMbulge = qlf.dNdlogMbulge

dNdlogL = qlf.dNdlogL
SBHARs = qlf.SBHARs



qlf_orig = QLF_orig(z, b)
qlf_orig.get_Mbh(Mb_crit, slope_low, norm_from_local, approx_local=True, norm_local = 11+norm_of_local)
qlf_orig.get_dNdlnL(L, [pre, post])
logMbh_orig = qlf_orig.BHBins
logMstar_orig = qlf_orig.StellBins
dNdlogL_orig = qlf_orig.dNdlogL
SBHARs_orig = qlf_orig.SBHARs


lumsshen = L
xshen = 10**L*3.8e33

dens, stanave, stanab, stanb, _ = Shen_fit_uncer(z, lumsshen)



ax0 = fig.add_subplot(gs[0, 0])

ax0.plot(10**logMstar, 10**logMbulge, c='k')
ax0.set_xlabel(r'$\rmM_* (M_\odot$)')
ax0.set_ylabel(r'$\rmM_{bulge} (M_\odot$)')
ax0.set_xscale('log')
ax0.set_yscale('log')
ax0.set_xlim([1e7, 1e12])
ax0.set_ylim([1e5, 1e12])



ax1 = fig.add_subplot(gs[0, 1])

ax1.plot(10**logMstar, SSFRs, c='r', label=r'$\rmM=M_*$')
ax1.plot(10**logMbulge, SMBARs, c='b', label=r'$\rmM=M_{bulge}$')

ax1.set_xlabel(r'M ($\rmM_\odot$)')
ax1.set_ylabel(r'$\rm\dot{M}/M \ (yr^{-1}$)')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlim([1e7, 1e12])
ax1.legend()

ax2 = fig.add_subplot(gs[0, 2])

ax2.plot(10**logMstar, np.log10(dNdlogMstar), c='r', label=r'$\rmM=M_*$')
ax2.plot(10**logMbulge, np.log10(dNdlogMbulge), c='b', label=r'$\rmM=M_{bulge}$')

ax2.set_xlabel(r'M ($\rmM_\odot$)')
ax2.set_ylabel(r'$\log_{10} \Phi (\rmMpc^{-3} \log_{10} [M]^{-1})$')
ax2.set_xscale('log')
ax2.set_xlim([1e7, 1e12])
ax2.set_ylim([-7,0])
ax2.legend()

ax3 = fig.add_subplot(gs[1, 0])

ax3.plot(10**logMbulge, 10**logMbh, c='k', label=r'With $\rmM_{bulge}$ relation.')
ax3.plot(10**logMstar_orig, 10**logMbh_orig, c='r', lw=1, label=r'Without $\rmM_{bulge}$ relation.')
ax3.plot(10**logMbulge, 10**(logMbulge-2.8), c='k', linestyle='dashed', label='~Observed')
ax3.set_xlabel(r'$\rmM_{bulge} (M_\odot$)')
ax3.set_ylabel(r'$\rmM_{BH} (M_\odot$)')
ax3.set_xlim((10**8, 10**12))
ax3.set_ylim((10**4,10**9))
ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.legend()


ax4 = fig.add_subplot(gs[1, 1])

ax4.plot(10**logMbh, SBHARs*3.154e7, c='k', label=r'With $\rmM_{bulge}$ relation.')
ax4.plot(10**logMbh_orig, SBHARs_orig*3.154e7, c='r', label=r'Without $\rmM_{bulge}$ relation.')
ax4.set_xlabel(r'$\rmM_{BH} (M_\odot$)')
ax4.set_ylabel(r'$\rm\dot{M}_{BH}/M_{BH} \ (yr^{-1}$)')
ax4.set_xscale('log')
ax4.set_yscale('log')
ax4.set_xlim((10**4,10**9))
ax4.legend()


ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(10**L*3.8e33, np.log10(dNdlogL), c='k', label=r'With $\rmM_{bulge}$ relation.')
ax5.plot(10**L*3.8e33, np.log10(dNdlogL_orig), c='r', label=r'Without $\rmM_{bulge}$ relation.')
ax5.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
ax5.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)

ax5.axvline(10**8.95*3.8e33,c='k',linestyle='dotted')
ax5.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')

ax5.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
ax5.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
ax5.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
ax5.text(10**48.75,-3,'z = '+str(z),fontsize = 12)
ax5.set_xscale('log')
ax5.legend()




plt.tight_layout()





FigureCanvasNbAgg()

## Exploring fits to the QLF with bulge relation.

### Seperate redshift values no limitations.

In [48]:
FILE="chi2_output/chi2_fiducial_2P_z0.5.h5py"
f = h5py.File(FILE, "r") 
siglnX2 = f['siglnX2'][:]
siglnX1 = f['siglnX1'][:]
chi2_grid = f['chi2_grid'][:].T
f.close()

for index, value in np.ndenumerate(chi2_grid):
    if siglnX2[index[3]] > siglnX1[index[4]]:
        chi2_grid[index] = 1e10
        
null = np.where(chi2_grid == 1e10)

In [49]:
def CHI2_SHEN_6param(z, chi2_grid):

    fig = plt.figure(figsize=(15,12))
    gs = gridspec.GridSpec(5, 5)
    
    minval = np.amin(chi2_grid)
    minind = np.where(chi2_grid == minval)

    bestlocal = norm_of_local[minind[0][0]] 
    bestnorm = norm_from_local[minind[1][0]]
    bestslope = slope_low[minind[2][0]]
    bestpost = siglnX2[minind[3][0]]
    bestpre = siglnX1[minind[4][0]]
    bestcrit = logMstar0[minind[5][0]]   
                     
    axj = [0,1,2,3,4,0,1,2,3,0,1,2,0,1,0]
    axi = [4,4,4,4,4,3,3,3,3,2,2,2,1,1,0]
    axes_l = []
    props = dict(boxstyle='square', facecolor='white', alpha=1)
    
    extents = [\
               [slope_low[0], slope_low [-1], norm_of_local[-1], norm_of_local[0]],\
               [norm_from_local[0], norm_from_local[-1], norm_of_local[-1], norm_of_local[0]],\
               [logMstar0[0], logMstar0[-1], norm_of_local[-1], norm_of_local[0]],\
               [siglnX1[0], siglnX1[-1], norm_of_local[-1], norm_of_local[0]],\
               [siglnX2[0], siglnX2[-1], norm_of_local[-1], norm_of_local[0]],\
               [slope_low[0], slope_low[-1], siglnX2[-1], siglnX2[0]],\
               [norm_from_local[0], norm_from_local[-1], siglnX2[-1], siglnX2[0]],\
               [logMstar0[0], logMstar0[-1], siglnX2[-1], siglnX2[0]],\
               [siglnX1[0], siglnX1[-1], siglnX2[-1], siglnX2[0]],\
               [slope_low[0], slope_low[-1], siglnX1[-1], siglnX1[0]],\
               [norm_from_local[0], norm_from_local[-1], siglnX1[-1], siglnX1[0]],\
               [logMstar0[0], logMstar0[-1], siglnX1[-1], siglnX1[0]],\
               [slope_low[0], slope_low[-1], logMstar0[-1], logMstar0[0]],\
               [norm_from_local[0], norm_from_local[-1], logMstar0[-1], logMstar0[0]],\
               [slope_low[0], slope_low[-1], norm_from_local[-1], norm_from_local[0]]\
              ]
    
    maj_axes = [[0.5, 0.2], [0.5, 0.2], [0.5, 0.2], [0.5, 0.2], [0.5, 0.2],\
                [0.5, 0.5], [0.5, 0.5], [0.5, 0.5], [0.5, 0.5],\
                [0.5, 0.5], [0.5, 0.5], [0.5, 0.5],\
                [0.5, 0.5], [0.5, 0.5],\
                [0.5, 0.5]]
    
    min_axes = [[0.25, 0.1],[0.25, 0.1],[0.25, 0.1],[0.25, 0.1],[0.25, 0.1],\
                [0.25, 0.25], [0.25, 0.25], [0.25, 0.25], [0.25, 0.25],\
                [0.25, 0.25], [0.25, 0.25], [0.25, 0.25],\
                [0.25, 0.25], [0.25, 0.25],\
                [0.25, 0.25]]
    
    bl = ''  
    slopel = r'Pre-disk slope'
    norml = r'Pre-disk norm (log)'
    prel = r'$\sigma_{\ln{X}pre}$'
    postl = r'$\sigma_{\ln{X}post}$'
    critl = r'Transition point (log)'
    locall = r'Post-disk norm (log)'
    xlabels = [slopel, norml, critl, prel, postl, bl, bl, bl, bl, bl, bl, bl, bl, bl, bl]
    ylabels = [locall, bl, bl, bl, bl, postl, bl, bl, bl, prel, bl, bl, critl, bl, norml]
    axess = [[0.1,1.5,np.log10(0.001),np.log10(0.005)],[0.5,3,np.log10(0.001),np.log10(0.005)],[9,11.5,np.log10(0.001),np.log10(0.005)],[1,4,np.log10(0.001),np.log10(0.005)],[1,4,np.log10(0.001),np.log10(0.005)],\
             [0.1,1.5,1,4],[0.5,3,1,4],[9,11.5,1,4],[1,4,1,4],\
             [0.1,1.5,1,4],[0.5,3,1,4],[9,11.5,1,4],\
             [0.1,1.5,9,11.5],[0.5,3,9,11.5],\
             [0.1,1.5,0.5,3]]
    slopet = [0.25, 0.5, 0.75, 1.0, 1.25]
    sums = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14]
    for ind, i, j in zip(range(15), axi, axj):   
        
        ### set up axis and label info
        ylabel = ylabels[ind]
        xlabel = xlabels[ind]
        axis = axess[ind]
        extent = extents[ind]

                     
        ax = fig.add_subplot(gs[i, j])
        axes_l += [ax]
        ax.set_xlabel(xlabel,fontsize=12)
        ax.set_ylabel(ylabel,fontsize=12)
        ax.axis(axis)
        
        ax.xaxis.set_major_locator(ticker.MultipleLocator(maj_axes[ind][0]))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(min_axes[ind][0]))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(maj_axes[ind][1]))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(min_axes[ind][1]))
        
        ax.tick_params(axis='both', which='both', labelsize=8, direction='in')
        
        
        if sums[ind] == 0:
            chi2_proj = np.zeros(chi2_grid[:,0,:,0,0,0].shape)
            for i in range(len(chi2_grid[:,0,0,0,0,0])):
                for j in range(len(chi2_grid[0,0,:,0,0,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[i, :, j, :, :,:])
            grid2d = chi2_proj
            ax.scatter(bestslope, bestlocal, marker = 'x', c='r',s=200)
            
        elif sums[ind] == 1:
            chi2_proj = np.zeros(chi2_grid[:,:,0,0,0,0].shape)
            for i in range(len(chi2_grid[:,0,0,0,0,0])):
                for j in range(len(chi2_grid[0,:,0,0,0,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[i, j, :, :, :, :])
            grid2d = chi2_proj
            ax.scatter(bestnorm, bestlocal, marker = 'x', c='r',s=200)
            
        elif sums[ind] == 2:
            chi2_proj = np.zeros(chi2_grid[:,0,0,0,0,:].shape)
            for i in range(len(chi2_grid[:,0,0,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,0,0,:])):
                    chi2_proj[i, j] = np.amin(chi2_grid[i, :, :, :, :, j])
            grid2d = chi2_proj
            ax.scatter(bestcrit, bestlocal, marker = 'x', c='r',s=200)
            
        elif sums[ind] == 3:
            chi2_proj = np.zeros(chi2_grid[:,0,0,0,:,0].shape)
            for i in range(len(chi2_grid[:,0,0,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,0,:,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[i, :, :, :, j, :])
            grid2d = chi2_proj
            ax.scatter(bestpre, bestlocal, marker = 'x', c='r',s=200)
    
        elif sums[ind] == 4:
            chi2_proj = np.zeros(chi2_grid[:,0,0,:,0,0].shape)
            for i in range(len(chi2_grid[:,0,0,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,:,0,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[i, :, :, j, :, :])
            grid2d = chi2_proj
            ax.scatter(bestpost, bestlocal, marker = 'x', c='r',s=200)
    
        elif sums[ind] == 5:
            chi2_proj = np.zeros(chi2_grid[0,0,:,:,0,0].shape)
            for i in range(len(chi2_grid[0,0,:,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,:,0,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, :, i, j, :, :])
            grid2d = chi2_proj.T
            ax.scatter(bestslope, bestpost, marker = 'x', c='r',s=200)
    
        elif sums[ind] == 6:
            chi2_proj = np.zeros(chi2_grid[0,:,0,:,0,0].shape)
            for i in range(len(chi2_grid[0,:,0,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,:,0,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, i, :, j, :, :])
            grid2d = chi2_proj.T
            ax.scatter(bestnorm, bestpost, marker = 'x', c='r',s=200)
    
        elif sums[ind] == 7:
            chi2_proj = np.zeros(chi2_grid[0,0,0,:,0,:].shape)
            for i in range(len(chi2_grid[0,0,0,:,0,0])):
                for j in range(len(chi2_grid[0,0,0,0,0,:])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, :, :, i, :, j])
            grid2d = chi2_proj
            ax.scatter(bestcrit, bestpost, marker = 'x', c='r',s=200)
    
        elif sums[ind] == 8:
            chi2_proj = np.zeros(chi2_grid[0,0,0,:,:,0].shape)
            for i in range(len(chi2_grid[0,0,0,:,0,0])):
                for j in range(len(chi2_grid[0,0,0,0,:,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, :, :, i, j, :])
            grid2d = chi2_proj
            ax.scatter(bestpre, bestpost, marker = 'x', c='r',s=200)
            
        elif sums[ind] == 9:
            chi2_proj = np.zeros(chi2_grid[0,0,:,0,:,0].shape)
            for i in range(len(chi2_grid[0,0,:,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,0,:,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, :, i, :, j, :])
            grid2d = chi2_proj.T
            ax.scatter(bestslope, bestpre, marker = 'x', c='r',s=200)
    
        elif sums[ind] == 10:
            chi2_proj = np.zeros(chi2_grid[0,:,0,0,:,0].shape)
            for i in range(len(chi2_grid[0,:,0,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,0,:,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, i, :, :, j, :])
            grid2d = chi2_proj.T
            ax.scatter(bestnorm, bestpre, marker = 'x', c='r',s=200)
    
        elif sums[ind] == 11:
            chi2_proj = np.zeros(chi2_grid[0,0,0,0,:,:].shape)
            for i in range(len(chi2_grid[0,0,0,0,:,0])):
                for j in range(len(chi2_grid[0,0,0,0,0,:])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, :, :, :, i, j])
            grid2d = chi2_proj
            ax.scatter(bestcrit, bestpre, marker = 'x', c='r',s=200)
    
        elif sums[ind] == 12:
            chi2_proj = np.zeros(chi2_grid[0,0,:,0,0,:].shape)
            for i in range(len(chi2_grid[0,0,:,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,0,0,:])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, :, i, :, :, j])
            grid2d = chi2_proj.T
            ax.scatter(bestslope, bestcrit, marker = 'x', c='r',s=200)
            
        elif sums[ind] == 13:
            chi2_proj = np.zeros(chi2_grid[0,:,0,0,0,:].shape)
            for i in range(len(chi2_grid[0,:,0,0,0,0])):
                for j in range(len(chi2_grid[0,0,0,0,0,:])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, i, :, :, :, j])
            grid2d = chi2_proj.T
            ax.scatter(bestnorm, bestcrit, marker = 'x', c='r',s=200)
            
        elif sums[ind] == 14: 
            chi2_proj = np.zeros(chi2_grid[0,:,:,0,0,0].shape)
            for i in range(len(chi2_grid[0,:,0,0,0,0])):
                for j in range(len(chi2_grid[0,0,:,0,0,0])):
                    chi2_proj[i, j] = np.amin(chi2_grid[:, i, j, :, :, :])
            grid2d = chi2_proj
            ax.scatter(bestslope, bestnorm, marker = 'x', c='r',s=200)
    
        
        norm = colors.Normalize(vmin = minval, vmax = np.amax(grid2d[grid2d != 1e10]))

        
        grid2d[grid2d == 1e10] = np.nan
        
        cmap = matplotlib.cm.get_cmap('gist_stern_r')
        cmap.set_bad(color='gray')
        img = ax.imshow(grid2d, cmap = cmap, interpolation = 'nearest', aspect='auto', extent = extent, norm=norm, alpha=0.85)
        
    print(f'Best crit = {bestcrit:.2f},  Best pre-disk = {bestpre:.2f}, Best post-disk = {bestpost:.2f},  Best slope = {bestslope:.2f},  Best norm = {bestnorm:.2f},Best local norm = {bestlocal:.2f}')

    plt.tight_layout()
    plt.subplots_adjust(wspace=0.1, hspace=0.1)
    cbar = plt.colorbar(img, label=r'', pad = 0.01, ax = axes_l).set_label(label=r'$\chi^2$',size=12)
    
    plt.setp(axes_l[5].get_xticklabels(), visible=False)
    plt.setp(axes_l[6].get_xticklabels(), visible=False)
    plt.setp(axes_l[7].get_xticklabels(), visible=False)
    plt.setp(axes_l[8].get_xticklabels(), visible=False)
    plt.setp(axes_l[9].get_xticklabels(), visible=False)
    plt.setp(axes_l[10].get_xticklabels(), visible=False)
    plt.setp(axes_l[11].get_xticklabels(), visible=False)
    plt.setp(axes_l[12].get_xticklabels(), visible=False)
    plt.setp(axes_l[13].get_xticklabels(), visible=False)
    plt.setp(axes_l[14].get_xticklabels(), visible=False)
    
    plt.setp(axes_l[1].get_yticklabels(), visible=False)
    plt.setp(axes_l[2].get_yticklabels(), visible=False)
    plt.setp(axes_l[3].get_yticklabels(), visible=False)
    plt.setp(axes_l[4].get_yticklabels(), visible=False)
    plt.setp(axes_l[6].get_yticklabels(), visible=False)
    plt.setp(axes_l[7].get_yticklabels(), visible=False)
    plt.setp(axes_l[8].get_yticklabels(), visible=False)
    plt.setp(axes_l[10].get_yticklabels(), visible=False)
    plt.setp(axes_l[11].get_yticklabels(), visible=False)
    plt.setp(axes_l[13].get_yticklabels(), visible=False)
    
    axes_l[6].text(7.5,12,'z = '+str(z), bbox=props, fontsize=12)
    axes_l[8].plot(np.linspace(1.0,10.0,2),np.linspace(1.0,10.0,2), linestyle = 'dashed',c='gray')

    return fig


In [53]:
plt.close('all')

redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']

with PdfPages('plots/Mstar-Mbulge/CHI2_bv_fiducial-B1_2P.pdf') as pdf:
    for z in redshifts:
        f = h5py.File("chi2_output/chi2_fiducial-B1_2P_z"+z+".h5py", "r") 
        logMstar0 = f['logMb0'][:]
        siglnX2 = f['siglnX2'][:]
        siglnX1 = f['siglnX1'][:]
        slope_low = f['slope_low'][:]
        norm_from_local = f['norm_from_local'][:]
        norm_of_local = f['norm_of_local'][:]
        chi2_grid = f['chi2_grid'][:].T
        f.close()

        chi2_grid[null] = 1e10


        fig = CHI2_SHEN_6param(float(z), chi2_grid)

        pdf.savefig(fig)
          

FigureCanvasNbAgg()

Best crit = 10.62,  Best pre-disk = 2.40, Best post-disk = 1.40,  Best slope = 0.10,  Best norm = 2.50,Best local norm = -2.36


FigureCanvasNbAgg()

Best crit = 10.75,  Best pre-disk = 2.00, Best post-disk = 1.80,  Best slope = 0.35,  Best norm = 2.25,Best local norm = -2.30


FigureCanvasNbAgg()

Best crit = 10.75,  Best pre-disk = 2.20, Best post-disk = 1.40,  Best slope = 0.70,  Best norm = 2.00,Best local norm = -2.36


FigureCanvasNbAgg()

Best crit = 10.62,  Best pre-disk = 2.40, Best post-disk = 1.20,  Best slope = 0.25,  Best norm = 2.00,Best local norm = -2.30


FigureCanvasNbAgg()

Best crit = 10.50,  Best pre-disk = 2.60, Best post-disk = 1.20,  Best slope = 0.10,  Best norm = 3.00,Best local norm = -2.36


In [ ]:
plt.close('all')

redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']

with PdfPages('plots/Mstar-Mbulge/CHI2_bv_fiducial-B2_2P.pdf') as pdf:
    for z in redshifts:
        f = h5py.File("chi2_output/chi2_fiducial-B2_2P_z"+z+".h5py", "r") 
        logMstar0 = f['logMb0'][:]
        siglnX2 = f['siglnX2'][:]
        siglnX1 = f['siglnX1'][:]
        slope_low = f['slope_low'][:]
        norm_from_local = f['norm_from_local'][:]
        norm_of_local = f['norm_of_local'][:]
        chi2_grid = f['chi2_grid'][:].T
        f.close()

        chi2_grid[null] = 1e10


        fig = CHI2_SHEN_6param(float(z), chi2_grid)

        pdf.savefig(fig)
          

FigureCanvasNbAgg()

Best crit = 10.62,  Best pre-disk = 2.40, Best post-disk = 1.40,  Best slope = 0.10,  Best norm = 2.50,Best local norm = -2.36


FigureCanvasNbAgg()

Best crit = 10.75,  Best pre-disk = 2.00, Best post-disk = 1.80,  Best slope = 0.35,  Best norm = 2.25,Best local norm = -2.30


FigureCanvasNbAgg()

In [54]:
plt.close('all')

redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']

with PdfPages('plots/Mstar-Mbulge/CHI2_bv_fiducial-B3_2P.pdf') as pdf:
    for z in redshifts:
        f = h5py.File("chi2_output/chi2_fiducial-B3_2P_z"+z+".h5py", "r") 
        logMstar0 = f['logMb0'][:]
        siglnX2 = f['siglnX2'][:]
        siglnX1 = f['siglnX1'][:]
        slope_low = f['slope_low'][:]
        norm_from_local = f['norm_from_local'][:]
        norm_of_local = f['norm_of_local'][:]
        chi2_grid = f['chi2_grid'][:].T
        f.close()

        chi2_grid[null] = 1e10


        fig = CHI2_SHEN_6param(float(z), chi2_grid)

        pdf.savefig(fig)
          

FigureCanvasNbAgg()

Best crit = 11.38,  Best pre-disk = 2.60, Best post-disk = 2.00,  Best slope = 0.65,  Best norm = 0.50,Best local norm = -2.30


FigureCanvasNbAgg()

Best crit = 9.00,  Best pre-disk = 3.00, Best post-disk = 2.40,  Best slope = 0.50,  Best norm = 0.50,Best local norm = -2.30


FigureCanvasNbAgg()

Best crit = 9.00,  Best pre-disk = 3.80, Best post-disk = 2.20,  Best slope = 0.45,  Best norm = 0.50,Best local norm = -2.30


FigureCanvasNbAgg()

Best crit = 9.12,  Best pre-disk = 4.00, Best post-disk = 2.00,  Best slope = 0.25,  Best norm = 0.50,Best local norm = -2.30


FigureCanvasNbAgg()

Best crit = 9.00,  Best pre-disk = 4.00, Best post-disk = 2.00,  Best slope = 0.30,  Best norm = 0.50,Best local norm = -2.30


In [87]:
def Shen_fit_uncer(z, lums, ver): ###best fit data from Shen+2020

    def get_params(params):
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params

    def shen_func(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(z, [d0]) + C.chebval(1 + z, [0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_a(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_b(p):
        L = lums
        a0, a1, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = a0*zfrac**a1
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)

    params = {'a0':[0.85858, 0.03092, 0.02876], 'a1':[-0.26236, 0.02003, 0.01753], 'a2':[0.02105, 0.00136, 0.00113],\
        'b0':[2.54992, 0.01915, 0.02949], 'b1':[-1.04735, 0.01815, 0.02999], 'b2':[1.13277, 0.01988, 0.03891],\
        'c0':[13.01297, 0.00943, 0.01354], 'c1':[-0.57587, 0.00205, 0.00261], 'c2':[0.45361, 0.00290, 0.00434],\
        'd0':[-3.53138, 0.02694, 0.02690], 'd1':[-0.39961, 0.00871, 0.00896]}
    
    params_a = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    
    params_b = {'a0':[0.3653, 0.0115, 0.0114], 'a1':[-0.6006, 0.0422, 0.0417],\
        'b0':[2.4709,0.0163,0.0169], 'b1':[-0.9963,0.0167,0.0161], 'b2':[1.0716, 0.0180, 0.0181],\
        'c0':[12.9656,0.0092,0.0089], 'c1':[-0.5758,0.0020,0.0019], 'c2':[0.4698,0.0025,0.0026],\
        'd0':[-3.6276,0.0209, 0.0203], 'd1':[-0.3444,0.0063,0.0061]}
    
    
    if ver == 'orig':
        param_list = np.array([params[i] for i in params])
        NUM = int(1e4)
        rand_params = get_params(params)
        ys = np.apply_along_axis(shen_func, 1, rand_params).T
        ya = shen_func(param_list[:,0])
    elif ver == 'a':
        param_list = np.array([params_a[i] for i in params_a])
        NUM = int(1e4)
        rand_params = get_params(params_a)
        ys = np.apply_along_axis(shen_func_a, 1, rand_params).T
        ya = shen_func_a(param_list[:,0]) 
    elif ver == 'b':
        param_list = np.array([params_b[i] for i in params_b])
        NUM = int(1e4)
        rand_params = get_params(params_b)
        ys = np.apply_along_axis(shen_func_b, 1, rand_params).T
        ya = shen_func_b(param_list[:,0]) 

    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya

    return ya, std_ave, std_abv, std_blw





def QLFwShen_test(ax, fit_params = None, z = 0.0, name = 'z0-QLF-v-Shen.pdf', Hopkins = False, approx_local=True, norm_local = 8.2):
   

    ### what fit params are we using
    if not fit_params:
        siglnM = 0.7
        bins = 0.005
        start = 10.0
        siglnX = [3.0, 2.0]
        lums = np.linspace(5,18,200)
    else:
        siglnM, bins, start, siglnX, slope, norm, lums = fit_params
    
    qlf = QLF(z, bins)
    qlf.get_Mbh(start, slope, norm, norm_local = norm_local)

    m = qlf.slopes

    qlf.get_dNdlnL(lums, siglnX)
    lumsp = 10**lums*3.8e33
    prea = np.zeros(len(lumsp))
    posta = np.zeros(len(lumsp))

    for i, pre, m in zip(np.transpose(qlf.intvals), qlf.pre, qlf.slopes):
        dens = i
        if pre == True:
            prea += dens
        else:
            posta += dens

    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF')
    
    pars = 6
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.5, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -10, np.log10(temp * np.log(10)), color = c, alpha = 0.2)
        mass_begin += 1

    lumsshen = lums
    xshen = 10**lumsshen*3.8e33

    dens, stanave, stanab, stanb = Shen_fit_uncer(z, lumsshen, 'a')
    
    ax.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)
    
    
    ### formatting and save
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-3,'z = '+str(z),fontsize = 12)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    

def bestfit_QLF_6param_multiz(z, FILE, pdf):
    f = h5py.File(FILE, "r") 
    logMb0 = f['logMb0'][:]
    siglnX2 = f['siglnX2'][:]
    siglnX1 = f['siglnX1'][:]
    slope_low = f['slope_low'][:]
    norm_from_local = f['norm_from_local'][:]
    norm_of_local = f['norm_of_local'][:]
    chi23d = f['chi2_grid'][:].T
    f.close()

    for index, value in np.ndenumerate(chi23d):
        if siglnX2[index[3]] > siglnX1[index[4]]:
            chi23d[index] = 1e10

    minval = np.amin(chi23d)
    minind = np.where(chi23d == minval)

    bestlocal = norm_of_local[minind[0][0]] 
    bestnorm = norm_from_local[minind[1][0]]
    bestslope = slope_low[minind[2][0]]
    bestpost = siglnX2[minind[3][0]]
    bestpre = siglnX1[minind[4][0]]
    bestcrit = logMb0[minind[5][0]] 

    print(f'\nShen best fits (z = {z}): minval =',minval)
    print(f'Best crit = {bestcrit:.2f},  Best pre-disk = {bestpre:.2f}, Best post-disk = {bestpost:.2f},  Best slope = {bestslope:.2f},  Best norm = {bestnorm:.2f},   Best local norm = {bestlocal:.2f}\n ')

    fig, (ax, ax4) = plt.subplots(1,2,figsize=(10,3.5))

    fit_params = [0.2, 0.005, bestcrit, [bestpre, bestpost], bestslope, bestnorm, np.linspace(5,18,200)]

    QLFwShen_test(ax, z=z, fit_params=fit_params, approx_local = True, norm_local = 11+bestlocal)

    ax.axvline(10**7.5*3.8e33,c='r',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    ax.text(10**43.8,-2.3, r'min($\chi^2$) = '+f'{minval:.2f}', fontsize = 10)

    pfs = 8
    ax4.text(10**7.2, 10**8.2, r'$\rmM_{Bcrit} = 10^{'+f'{bestcrit:.2f}'+'} M_\odot$', fontsize = pfs)
    ax4.text(10**7.2, 10**7.7, r'$\sigma_{\rmpre\ \ln X} = $'+f'{bestpre:.2f}', fontsize = pfs)
    ax4.text(10**7.2, 10**7.2, r'$\sigma_{\rmpost\ \ln X} = $'+f'{bestpost:.2f}', fontsize = pfs)
    ax4.text(10**10.2, 10**5.2, r'$\frac{\rmd\ln M_{BH}}{d\ln M_{bulge}} = $'+f'{bestslope:.3f}', fontsize = pfs)
    ax4.text(10**10.2, 10**4.7, r'$\rmM_{BHnorm} = 10^{'+f'{bestnorm:.2f}'+'} M_\odot$', fontsize = pfs)
    ax4.text(10**10.2, 10**4.2, r'post $\frac{\rmM_{BH}}{M_{bulge}} = $'+f'{10**bestlocal:.4f}', fontsize = pfs)




    pars = 6
    cs = list(cm.Greens(np.arange(pars) / pars) ) 


    qlf = QLF(z, fit_params[1])
    qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], norm_local = 11+bestlocal)
    qlf.get_dNdlnL(fit_params[-1], fit_params[3])


    ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
    ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='~Observed')
    ax4.set_xlabel('$M_{bulge}(M_\odot)$',fontsize=12)
    ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
    ax4.set_yscale('log')
    ax4.set_xscale('log')
    ax4.set_xlim([10**7,10**12.5])
    ax4.set_ylim([10**3, 10**10])
    ax4.legend(fontsize = 8, framealpha=1)
    ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')
    for c, mass in zip(cs, [7,8,9,10,11,12,13]):
        ax4.axvline(10**mass, ls = 'dotted', c=c)
        ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)

    plt.suptitle('6 Param SHEN Best Fit')
    plt.tight_layout()

    pdf.savefig(fig)



In [88]:
with PdfPages('plots/bestfit_plots/ALLZ_bulge_medres_QLF_bestfit_6param-SHEN-eL.pdf') as pdf:
    for z in ['0.0', '0.5', '1.0', '1.5', '2.0', '2.5', '3.0', '3.5', '4.0']:
        bestfit_QLF_6param_multiz(z=float(z), FILE="output_bulge/chi2_SHEN_6param_bulge_eL_adjreso_z"+z+".h5py", pdf=pdf)



Shen best fits (z = 0.0): minval = 0.05840879092532078
Best crit = 10.16,  Best pre-disk = 3.79, Best post-disk = 2.07,  Best slope = 0.08,  Best norm = 2.29,   Best local norm = -2.95
 


FigureCanvasNbAgg()


Shen best fits (z = 0.5): minval = 0.9407246404943247
Best crit = 10.16,  Best pre-disk = 2.50, Best post-disk = 1.86,  Best slope = 0.35,  Best norm = 1.75,   Best local norm = -2.55
 


FigureCanvasNbAgg()


Shen best fits (z = 1.0): minval = 1.3514483508063906
Best crit = 10.53,  Best pre-disk = 1.86, Best post-disk = 1.86,  Best slope = 0.69,  Best norm = 1.75,   Best local norm = -2.35
 


...

FigureCanvasNbAgg()


Shen best fits (z = 3.5): minval = 0.5045213569029382
Best crit = 10.53,  Best pre-disk = 2.07, Best post-disk = 1.43,  Best slope = 1.30,  Best norm = 1.21,   Best local norm = -2.35
 


FigureCanvasNbAgg()


Shen best fits (z = 4.0): minval = 0.26014132995734907
Best crit = 10.53,  Best pre-disk = 2.07, Best post-disk = 1.43,  Best slope = 1.23,  Best norm = 1.04,   Best local norm = -2.35
 


FigureCanvasNbAgg()